# Food Supply Chain Sales & Profitability Analysis

**Business question:** Which product categories and regions drive profitability, and where should a food manufacturer focus to improve margins and reduce supply chain costs?

This notebook loads the Superstore dataset, maps retail categories to a food manufacturer's portfolio (Beverages, Snacks, Perishables), cleans the data, and produces the KPIs, charts and dashboard data files.

In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
OUT_CHARTS = ROOT / "outputs" / "charts"
OUT_DOCS = ROOT / "docs" / "data"
OUT_CHARTS.mkdir(parents=True, exist_ok=True)
OUT_DOCS.mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)

print("Repo root:", ROOT)

Repo root: /home/runner/work/food-supply-chain-sales-dashboard/food-supply-chain-sales-dashboard


## 1. Load the data
The raw file is `data/superstore.csv`. Dates are parsed, and the retail `Product Category` is mapped to a food portfolio so the analysis reads like a food manufacturer's distribution network.

In [2]:
df = pd.read_csv(DATA / "superstore.csv")
print("Raw shape:", df.shape)
df.head()

Raw shape: (8399, 21)


,Row ID,Order ID,Order Date,Order Priority,Order Quantity,Sales,Discount,Ship Mode,Profit,Unit Price,...,Customer Name,Province,Region,Customer Segment,Product Category,Product Sub-Category,Product Name,Product Container,Product Base Margin,Ship Date
0,1,3,10/13/2010,Low,6,261.5400,0.04,Regular Air,-213.25,38.94,...,Muhammed MacIntyre,Nunavut,Nunavut,Small Business,Office Supplies,Storage & Organization,"Eldon Base for stackable storage shelf, platinum",Large Box,0.80,10/20/2010
1,49,293,10/1/2012,High,49,10123.0200,0.07,Delivery Truck,457.81,208.16,...,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Jumbo Drum,0.58,10/2/2012
2,50,293,10/1/2012,High,27,244.5700,0.01,Regular Air,46.71,8.69,...,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Binders and Binder Accessories,"Cardinal Slant-D� Ring Binder, Heavy Gauge Vinyl",Small Box,0.39,10/3/2012
3,80,483,7/10/2011,High,30,4965.7595,0.08,Regular Air,1198.97,195.99,...,Clay Rozendal,Nunavut,Nunavut,Corporate,Technology,Telephones and Communication,R380,Small Box,0.58,7/12/2011
4,85,515,8/28/2010,Not Specified,19,394.2700,0.08,Regular Air,30.94,21.78,...,Carlos Soltero,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,Holmes HEPA Air Purifier,Medium Box,0.50,8/30/2010


In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

CATEGORY_MAP = {
    "Technology": "Beverages",
    "Furniture": "Snacks",
    "Office Supplies": "Perishables",
}
df["Food Category"] = df["Product Category"].map(CATEGORY_MAP).fillna("Other")

## 2. Clean the data
- Drop rows without sales, profit or order date.
- Coerce numeric columns.
- Remove duplicate order lines.
- Clip discounts to a sane 0–1 range.

In [4]:
numeric_cols = ["Order Quantity", "Sales", "Discount", "Profit", "Unit Price",
                "Shipping Cost", "Product Base Margin"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=["Sales", "Profit", "Order Date"])
df = df.drop_duplicates(subset=["Row ID"])
df["Discount"] = df["Discount"].clip(lower=0, upper=1)
df["Shipping Cost"] = df["Shipping Cost"].fillna(0)
df["Year"] = df["Order Date"].dt.year
print(f"Dropped {before - len(df)} rows during cleaning. Clean shape: {df.shape}")

Dropped 0 rows during cleaning. Clean shape: (8399, 23)


## 3. KPI overview
Headline numbers for the dashboard: total sales, profit margin, average discount, and year-over-year sales growth.

In [5]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales * 100
avg_discount = df["Discount"].mean() * 100

yearly = df.groupby("Year")["Sales"].sum().sort_index()
if len(yearly) >= 2:
    sales_growth = (yearly.iloc[-1] / yearly.iloc[-2] - 1) * 100
else:
    sales_growth = 0.0

print(f"Total Sales:      ${total_sales:,.2f}")
print(f"Total Profit:     ${total_profit:,.2f}")
print(f"Profit Margin:    {profit_margin:.2f}%")
print(f"Avg Discount:     {avg_discount:.2f}%")
print(f"Sales Growth YoY: {sales_growth:.2f}%")

Total Sales:      $14,915,600.82
Total Profit:     $1,521,767.98
Profit Margin:    10.20%
Avg Discount:     4.97%
Sales Growth YoY: 8.24%


## 4. Category profitability
Which food categories drive sales and profit? This answers the first half of the business question.

In [6]:
cat = (df.groupby("Food Category")
         .agg(total_sales=("Sales", "sum"),
              total_profit=("Profit", "sum"),
              avg_discount=("Discount", "mean"),
              order_lines=("Row ID", "count"))
         .reset_index())
cat["profit_margin_pct"] = cat["total_profit"] / cat["total_sales"] * 100
cat = cat.sort_values("total_sales", ascending=False)
cat

,Food Category,total_sales,total_profit,avg_discount,order_lines,profit_margin_pct
0,Beverages,5984248.182,886313.52,0.048746,2065,14.810775
2,Snacks,5178590.542,117433.03,0.049287,1724,2.267664
1,Perishables,3752762.100,518021.43,0.050230,4610,13.803738


In [7]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(cat))
ax.bar(x - 0.2, cat["total_sales"], width=0.4, label="Sales", color="#2E86AB")
ax.bar(x + 0.2, cat["total_profit"], width=0.4, label="Profit", color="#F18F01")
ax.set_xticks(x)
ax.set_xticklabels(cat["Food Category"])
ax.set_ylabel("CAD")
ax.set_title("Sales vs Profit by Food Category")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "category_sales_profit.png", dpi=120)
plt.close(fig)

## 5. Regional profitability
Profit per province shows where the distribution network is strongest and weakest.

In [8]:
region = (df.groupby("Province")
            .agg(total_sales=("Sales", "sum"),
                 total_profit=("Profit", "sum"),
                 avg_discount=("Discount", "mean"),
                 shipping_cost=("Shipping Cost", "sum"))
            .reset_index())
region["profit_margin_pct"] = region["total_profit"] / region["total_sales"] * 100
region = region.sort_values("total_profit", ascending=False)
region.head(10)

,Province,total_sales,total_profit,avg_discount,shipping_cost,profit_margin_pct
8,Ontario,3.063212e+06,346868.54,0.049146,23475.26,11.323685
11,Saskachewan,1.464456e+06,184732.96,0.049321,10420.24,12.614444
0,Alberta,1.704791e+06,151946.48,0.048971,11118.67,8.912907
1,British Columbia,1.892758e+06,145062.13,0.048694,13464.48,7.664062
10,Quebec,1.510195e+06,140426.65,0.049808,10543.34,9.298577
2,Manitoba,1.372849e+06,136427.16,0.050643,10450.69,9.937523
3,New Brunswick,6.842115e+05,115351.94,0.048514,4046.41,16.859105
5,Northwest Territories,8.008473e+05,100653.08,0.050076,5418.93,12.568323
6,Nova Scotia,8.177294e+05,85361.87,0.051681,6480.42,10.438890
12,Yukon,9.758674e+05,73849.21,0.050812,7469.81,7.567546


In [9]:
top_regions = region.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_regions["Province"], top_regions["total_profit"], color="#3BB273")
ax.set_xlabel("Total Profit (CAD)")
ax.set_title("Top 10 Provinces by Profit")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "region_profit.png", dpi=120)
plt.close(fig)

## 6. Discount impact
Discounting is the classic margin killer. Bucketing orders by discount band shows the trade-off between volume and margin.

In [10]:
def discount_band(d):
    if d == 0:
        return "0% (No discount)"
    if d <= 0.05:
        return "1-5%"
    if d <= 0.10:
        return "6-10%"
    if d <= 0.20:
        return "11-20%"
    return "20%+"

df["Discount Band"] = df["Discount"].apply(discount_band)
disc = (df.groupby("Discount Band")
          .agg(total_sales=("Sales", "sum"),
               total_profit=("Profit", "sum"),
               order_lines=("Row ID", "count"),
               avg_profit=("Profit", "mean"))
          .reset_index())
disc["profit_margin_pct"] = disc["total_profit"] / disc["total_sales"] * 100
band_order = ["0% (No discount)", "1-5%", "6-10%", "11-20%", "20%+"]
disc["sort"] = disc["Discount Band"].map({b: i for i, b in enumerate(band_order)})
disc = disc.sort_values("sort").drop(columns="sort")
disc

,Discount Band,total_sales,total_profit,order_lines,avg_profit,profit_margin_pct
0,0% (No discount),1493748.218,188188.77,756,248.926944,12.598426
1,1-5%,6932776.815,828183.46,3906,212.028536,11.945913
4,6-10%,6486807.517,505879.74,3732,135.551913,7.798593
2,11-20%,1265.970,14.80,3,4.933333,1.169064
3,20%+,1002.304,-498.79,2,-249.395000,-49.764343


In [11]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(disc["Discount Band"], disc["profit_margin_pct"], marker="o", color="#C73E1D", linewidth=2)
ax.set_ylabel("Profit Margin (%)")
ax.set_title("Profit Margin by Discount Band")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_CHARTS / "discount_margin.png", dpi=120)
plt.close(fig)

## 7. Shipping mode cost efficiency
Shipping cost as a share of sales reveals which modes erode margin.

In [12]:
ship = (df.groupby("Ship Mode")
          .agg(total_sales=("Sales", "sum"),
               shipping_cost=("Shipping Cost", "sum"),
               avg_profit=("Profit", "mean"),
               order_lines=("Row ID", "count"))
          .reset_index())
ship["shipping_cost_pct"] = ship["shipping_cost"] / ship["total_sales"] * 100
ship = ship.sort_values("shipping_cost_pct", ascending=False)
ship

,Ship Mode,total_sales,shipping_cost,avg_profit,order_lines,shipping_cost_pct
0,Delivery Truck,6224878.718,51971.94,235.292208,1146,0.834907
1,Express Air,1184418.897,7850.91,149.979969,983,0.662849
2,Regular Air,7506303.209,48008.19,176.187049,6270,0.639572


In [13]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(ship["Ship Mode"], ship["shipping_cost_pct"], color="#6A4C93")
ax.set_ylabel("Shipping Cost (% of Sales)")
ax.set_title("Shipping Cost as % of Sales by Ship Mode")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "ship_mode_cost.png", dpi=120)
plt.close(fig)

## 8. Top products by profit
The top 5 products by profit, plus a count of loss-making products to flag for review.

In [14]:
prod = (df.groupby(["Product Name", "Food Category"])
          .agg(total_sales=("Sales", "sum"),
               total_profit=("Profit", "sum"))
          .reset_index())
prod["profit_margin_pct"] = prod["total_profit"] / prod["total_sales"] * 100
top_products = prod.sort_values("total_profit", ascending=False).head(5).reset_index(drop=True)
loss_making = int((prod["total_profit"] < 0).sum())
top_products

,Product Name,Food Category,total_sales,total_profit,profit_margin_pct
0,Global Troy� Executive Leather Low-Back Tilter,Snacks,275941.52,79284.78,28.732458
1,Hewlett Packard LaserJet 3310 Copier,Beverages,194880.35,73299.61,37.612622
2,"Hewlett-Packard cp1700 [D, PS] Series Color In...",Beverages,128100.43,55281.39,43.154726
3,GBC DocuBind 200 Manual Binding Machine,Perishables,120342.74,50462.53,41.932343
4,Canon PC940 Copier,Beverages,210910.68,47705.85,22.618983


In [15]:
fig, ax = plt.subplots(figsize=(10, 5))
short_names = [n[:30] + "..." if len(n) > 30 else n for n in top_products["Product Name"]]
ax.barh(short_names[::-1], top_products["total_profit"][::-1], color="#F18F01")
ax.set_xlabel("Total Profit (CAD)")
ax.set_title("Top 5 Products by Profit")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "top_products.png", dpi=120)
plt.close(fig)

## 9. Save summary and dashboard data
The flat `summary.json` feeds the KPI cards; each `docs/data/*.json` file feeds a dashboard chart.

In [16]:
summary = {
    "total_sales": round(float(total_sales), 2),
    "total_profit": round(float(total_profit), 2),
    "profit_margin_pct": round(float(profit_margin), 2),
    "avg_discount_pct": round(float(avg_discount), 2),
    "sales_growth_yoy_pct": round(float(sales_growth), 2),
    "order_lines": int(len(df)),
    "loss_making_products": loss_making,
    "top_category": str(cat.iloc[0]["Food Category"]),
    "top_province": str(region.iloc[0]["Province"]),
}
with open(ROOT / "outputs" / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "total_sales": 14915600.82,
  "total_profit": 1521767.98,
  "profit_margin_pct": 10.2,
  "avg_discount_pct": 4.97,
  "sales_growth_yoy_pct": 8.24,
  "order_lines": 8399,
  "loss_making_products": 553,
  "top_category": "Beverages",
  "top_province": "Ontario"
}


In [17]:
cat_out = cat[["Food Category", "total_sales", "total_profit", "profit_margin_pct"]].copy()
cat_out.columns = ["category", "total_sales", "total_profit", "profit_margin_pct"]
cat_out.to_json(OUT_DOCS / "category_profit.json", orient="records")

region_out = region[["Province", "total_profit", "profit_margin_pct"]].head(10).copy()
region_out.columns = ["province", "total_profit", "profit_margin_pct"]
region_out.to_json(OUT_DOCS / "region_profit.json", orient="records")

disc_out = disc[["Discount Band", "profit_margin_pct", "total_profit"]].copy()
disc_out.columns = ["discount_band", "profit_margin_pct", "total_profit"]
disc_out.to_json(OUT_DOCS / "discount_margin.json", orient="records")

ship_out = ship[["Ship Mode", "shipping_cost_pct"]].copy()
ship_out.columns = ["ship_mode", "shipping_cost_pct"]
ship_out.to_json(OUT_DOCS / "ship_mode.json", orient="records")

top_out = top_products[["Product Name", "total_profit"]].copy()
top_out.columns = ["product_name", "total_profit"]
top_out.to_json(OUT_DOCS / "top_products.json", orient="records")

print("Dashboard data files written to", OUT_DOCS)
print("Charts written to", OUT_CHARTS)

Dashboard data files written to /home/runner/work/food-supply-chain-sales-dashboard/food-supply-chain-sales-dashboard/docs/data
Charts written to /home/runner/work/food-supply-chain-sales-dashboard/food-supply-chain-sales-dashboard/outputs/charts
